# Objetivo da automação é realizar a atividade que a Dani realizava manualmente, conferindo os valores  

In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador"

### Normalizar e importar as Bibliotecas necessárias para as automações###

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("mes", "", "Data fechamento")
mes = dbutils.widgets.get("mes")

✅ Bibliotecas importadas e configurações aplicadas com sucesso!


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Exemplo informado


In [0]:
Fechamento_Trabalhista = f'/Volumes/databox/juridico_comum/arquivos/trabalhista/FECHAMENTO_TRABALHISTA/{mes} Fechamento Trabalhista.xlsx'

df_Fechamento_Trabalhista = pd.read_excel(
    Fechamento_Trabalhista,
    sheet_name="Base Fechamento", 
    header=1,
    engine="openpyxl"
)

In [0]:
#Converter o df consolidado do INSS para o pandas
df = spark.read.table("databox.juridico_comum.br_consolidado_INSS")
df_Base_Consolidada_INSS = df.toPandas()

### Verifica se existem ID PROCESSO duplicados no df_Fechamento_Trabalhista (Isso ocorre pois o precesso pode ter migrado de uma empresa para a outra)

In [0]:
# Encontra e exibe todas as linhas que têm 'ID PROCESSO' duplicado
duplicados = df_Fechamento_Trabalhista[df_Fechamento_Trabalhista.duplicated(subset=['ID PROCESSO'], keep=False)]

#Ordena para facilitar a vizão
print(duplicados.sort_values('ID PROCESSO'))


         LINHA  ID PROCESSO Área do Direito     Sub-área do Direito  \
2430   linha02      1235057     TRABALHISTA  CONTENCIOSO INDIVIDUAL   
6380   linha01      1235057     TRABALHISTA  CONTENCIOSO INDIVIDUAL   
24246  linha02      1260513     TRABALHISTA  CONTENCIOSO INDIVIDUAL   
27629  linha01      1260513     TRABALHISTA  CONTENCIOSO INDIVIDUAL   

      Grupo (M-1)          Empresa (M-1)   Grupo (M)  \
2430   GLOBEX GPA                    NaN  GLOBEX GPA   
6380   GLOBEX GPA  GRUPO CASAS BAHIA S/A  GLOBEX GPA   
24246       VVLOG                    NaN       VVLOG   
27629       VVLOG  GRUPO CASAS BAHIA S/A       VVLOG   

                                              Empresa (M) EMPRESA MAPA  \
2430   ASAP LOG LOGÍSTICA E SOLUÇÕES LTDA. (ANTIGA VVLOG)         ASAP   
6380                                GRUPO CASAS BAHIA S/A          GCB   
24246  ASAP LOG LOGÍSTICA E SOLUÇÕES LTDA. (ANTIGA VVLOG)         ASAP   
27629                               GRUPO CASAS BAHIA S/A          

### Cria os dicionários para converter as datas que estão em inglês para Português e Gera os meses automaticamente 

In [0]:
# Dicionário de tradução de meses abreviados (inglês → português)
meses_pt = {
    'JAN': 'JAN', 'FEB': 'FEV', 'MAR': 'MAR', 'APR': 'ABR',
    'MAY': 'MAI', 'JUN': 'JUN', 'JUL': 'JUL', 'AUG': 'AGO',
    'SEP': 'SET', 'OCT': 'OUT', 'NOV': 'NOV', 'DEC': 'DEZ'
}

In [0]:
def gerar_nome(col_base, data_ref):
    mes_en = data_ref.strftime('%b').upper()
    mes_pt = meses_pt.get(mes_en, mes_en)
    ano = data_ref.strftime('%y')
    
    # FORMATO ANTIGO: f"{col_base} - {mes_pt}-{ano}" -> "STATUS - AGO-25"
    # FORMATO NOVO: 
    return f"{col_base}_{mes_pt}_{ano}".upper()

In [0]:
# Datas de referência
hoje = datetime.now()
mes_atual_data = hoje
mes_anterior_data = (hoje.replace(day=1) - timedelta(days=1))  # último dia do mês anterior

In [0]:
# Colunas dinâmicas
col_status = gerar_nome("STATUS", mes_atual_data)
col_motivo = gerar_nome("MOTIVO", mes_atual_data)
col_provisao = gerar_nome("PROVISAO", mes_atual_data)
col_status_ant = gerar_nome("STATUS", mes_anterior_data)
col_movimento = gerar_nome("movimento", mes_atual_data)

### Cria o mapa (PROCV) e exclui os IDs duplicados e realiza o PROCV nas colunas(ID_PROCESSO, STATUS (M), MOTIVO MOVIMENTAÇÃO E PROVISÃO TOTAL (M))

In [0]:
#Cria o mapa de status, mas primeiro remove os IDs duplicados, mantendo apenas a primeira ocorrência
mapa_status = (
    df_Fechamento_Trabalhista
    .drop_duplicates(subset=['ID PROCESSO'], keep='last')
    .set_index('ID PROCESSO')[['STATUS (M)', 'MOTIVO_MOVIMENTAÇÃO', 'Provisão Total (M)']]
)

In [0]:
# 4. Preencher dinamicamente as colunas no df_base
df_Base_Consolidada_INSS[col_status] = df_Base_Consolidada_INSS['ID_PROCESSO'].map(mapa_status['STATUS (M)'])
df_Base_Consolidada_INSS[col_motivo] = df_Base_Consolidada_INSS['ID_PROCESSO'].map(mapa_status['MOTIVO_MOVIMENTAÇÃO'])
df_Base_Consolidada_INSS[col_provisao] = df_Base_Consolidada_INSS['ID_PROCESSO'].map(mapa_status['Provisão Total (M)'])

### Exibe os titulos das colunas para verificar se foi criada as colunas: Status do mês em questão

In [0]:
print(df_Fechamento_Trabalhista.columns.tolist())

['LINHA', 'ID PROCESSO', 'Área do Direito', 'Sub-área do Direito', 'Grupo (M-1)', 'Empresa (M-1)', 'Grupo (M)', 'Empresa (M)', 'EMPRESA MAPA', 'STATUS (M-1)', 'STATUS (M)', 'Centro de Custo (M-1)', 'Centro de Custo (M)', 'DATACADASTRO', 'Reabertura', 'Objeto Assunto/Cargo (M-1)', 'Sub Objeto Assunto/Cargo (M-1)', 'Objeto Assunto/Cargo (M)', 'Sub Objeto Assunto/Cargo (M)', 'Órgão ofensor (Fluxo) (M-1)', 'Órgão ofensor (Fluxo) (M)', 'Natureza Operacional (M-1)', 'Natureza Operacional (M)', 'Distribuição', 'Nº Processo', 'Escritorio', 'Vlr Causa', '% Sócio (M-1)', '% Empresa (M-1)', '% Sócio (M)', '% Empresa (M)', 'Provisão (M-1)', 'Correção  (M-1)', 'Provisão Total (M-1)', 'Classificação Mov. (M)', 'Provisão Mov. (M)', 'Correção Mov. (M)', 'Provisão Mov. Total (M)', 'Provisão Total (M)', 'Correção (M-1)', 'Correção (M)', 'Provisão Total Passivo (M)', 'Socio: Provisão (M-1)', 'Socio: Correção (M-1)', 'Socio: Provisão Total (M-1)', 'SOCIO: Classificação Mov. (M)', 'SOCIO: Provisão Mov. (M)

In [0]:
print(df_Base_Consolidada_INSS.columns.to_list())

['ID_PROCESSO', 'INSS_DECAIDO', 'DEMITIDO_POR_REESTRUTURACAO', 'CENTRO_DE_CUSTO_M', 'COD_EMPRESA', 'AREA_FUNCIONAL', 'ID_ORIGEM', 'CC_BLOQUEADO', 'STATUS_DEZ_22', 'STATUS_JAN_23', 'STATUS_FEV_23', 'STATUS_MAR_23', 'STATUS_ABR_23', 'STATUS_MAI_23', 'STATUS_JUN_23', 'STATUS_JUL_23', 'STATUS_AGO_23', 'STATUS_SET_23', 'STATUS_OUT_23', 'STATUS_NOV_23', 'STATUS_DEZ_23', 'STATUS_JAN_24', 'STATUS_FEV_24', 'STATUS_MAR_24', 'STATUS_ABR_24', 'STATUS_MAI_24', 'STATUS_JUN_24', 'STATUS_JUL_24', 'STATUS_AGO_24', 'STATUS_SET_24', 'STATUS_OUT_24', 'STATUS_NOV_24', 'STATUS_DEZ_24', 'STATUS_JAN_25', 'STATUS_FEV_25', 'STATUS_MAR_25', 'STATUS_ABR_25', 'STATUS_MAI_25', 'STATUS_JUN_25', 'STATUS_JUL_25', 'STATUS_AGO_25', 'STATUS_SET_25', 'STATUS_OUT_25', 'STATUS_NOV_25', 'STATUS_JAN_26', 'MOTIVO_JAN_26', 'PROVISAO_JAN_26']


### Cria as condições para exibição na tabela final

In [0]:
# 1. Se STATUS atual for nulo e o anterior não, copia o anterior
cond = pd.isna(df_Base_Consolidada_INSS[col_status]) & \
       df_Base_Consolidada_INSS[col_status_ant].notna()

df_Base_Consolidada_INSS.loc[cond, col_status] = df_Base_Consolidada_INSS.loc[cond, col_status_ant]

# 2. Se STATUS atual for ATIVO ou REATIVADO e o anterior não for nulo, mantém o anterior
cond = df_Base_Consolidada_INSS[col_status].isin(['ATIVO', 'REATIVADO']) & \
       df_Base_Consolidada_INSS[col_status].notna()
       
df_Base_Consolidada_INSS.loc[cond, col_status] = df_Base_Consolidada_INSS.loc[cond, col_status_ant]


---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File /databricks/python/lib/python3.11/site-packages/pandas/core/indexes/base.py:3802, in Index.get_loc(self, key, method, tolerance)
   3801 try:
-> 3802     return self._engine.get_loc(casted_key)
   3803 except KeyError as err:

File /databricks/python/lib/python3.11/site-packages/pandas/_libs/index.pyx:138, in pandas._libs.index.IndexEngine.get_loc()

File /databricks/python/lib/python3.11/site-packages/pandas/_libs/index.pyx:165, in pandas._libs.index.IndexEngine.get_loc()

File pandas/_libs/hashtable_class_helper.pxi:5745, in pandas._libs.hashtable.PyObjectHashTable.get_item()

File pandas/_libs/hashtable_class_helper.pxi:5753, in pandas._libs.hashtable.PyObjectHashTable.get_item()

KeyError: 'STATUS_DEZ_25'

The above exception was the direct cause of the following exception:

KeyError                                  Traceback (

In [0]:
#cond = df_Base_Consolidada_INSS['STATUS - JUL-25'].notna() & \
 #   df_Base_Consolidada_INSS['movimento- JUL-25'] == '4. Acordo/Vazio para Defesa'

cond = df_Base_Consolidada_INSS[col_motivo] == '4. Acordo/Vazio para Defesa'
df_Base_Consolidada_INSS.loc[cond, col_status] = df_Base_Consolidada_INSS.loc[cond, col_motivo]



In [0]:
# 4. Mapeia a provisão
df_Base_Consolidada_INSS['Valor da Provisão'] = df_Base_Consolidada_INSS['ID PROCESSO'].map(mapa_status['Provisão Total (M)'])

In [0]:
# 5. Se STATUS for ATIVO e Provisão == 0, altera
cond = (df_Base_Consolidada_INSS[col_status] == 'ATIVO') & \
       (df_Base_Consolidada_INSS['Valor da Provisão'] == 0)

df_Base_Consolidada_INSS.loc[cond, col_status] = 'ATIVO - BAIXA POR REVERSÃO DE PROVISÃO'


In [0]:
# 6. Se STATUS for REATIVADO e Provisão == 0, altera
cond = (df_Base_Consolidada_INSS[col_status] == 'REATIVADO') & \
       (df_Base_Consolidada_INSS['Valor da Provisão'] == 0)

df_Base_Consolidada_INSS.loc[cond, col_status] = 'REATIVADO - BAIXA POR REVERSÃO DE PROVISÃO'

### Dropa as Colunas temporarias

In [0]:
# Remover colunas de MOTIVO e PROVISAO do mês atual
colunas_para_dropar = [col_motivo, col_provisao]

df_Base_Consolidada_INSS.drop(columns=colunas_para_dropar, inplace=True)

### Exibe a coluna do mês vigente

In [0]:
# 3. Identificar a coluna de STATUS do mês atual (última coluna preenchida que começa com "STATUS - ")
coluna_status = [col for col in df_Base_Consolidada_INSS.columns if str(col).startswith("STATUS - ") and df_Base_Consolidada_INSS[col].notna().any()]
ultima_coluna_status = coluna_status[-1] if coluna_status else None

if ultima_coluna_status is None:
    raise ValueError("Nenhuma coluna STATUS encontrada.")

print(f"Coluna do mês vigente: {ultima_coluna_status}")

### Verifica se o valor do Somase bate com o valor feito pela Dani

In [0]:
col_valor = 'INSS DECAÍDO'

In [0]:
# Garante que a coluna de valores seja numérica
df_Base_Consolidada_INSS[col_valor] = pd.to_numeric(df_Base_Consolidada_INSS[col_valor], errors='coerce')v

In [0]:
filtro = df_Base_Consolidada_INSS[col_status].isin(['ATIVO', 'REATIVADO'])

In [0]:
# Soma apenas os valores filtrados
soma_valores = df_Base_Consolidada_INSS.loc[filtro, col_valor].sum()

In [0]:
print(f"Soma dos valores: {soma_valores}")

### Armazena o Arquivo em uma pasta (solução temporaria)

In [0]:
 #SALVA ARQUIVO   
    nome_arquivo ='Base_Consolidada_INSS.xlsx'
    caminho_local_temporario = f"/tmp/{nome_arquivo}"
    caminho_final_destino = f"/Volumes/databox/juridico_comum/arquivos/trabalhista/INSS_FINAL/{nome_arquivo}"

    # Caminho completo com diretório
    #caminho_completo = os.path.join(DIRETORIO_SAIDA, nome_arquivo)

    # Salva
   
    df_Base_Consolidada_INSS.to_excel(caminho_local_temporario, index=False)
    shutil.move(caminho_local_temporario, caminho_final_destino)

    print(f"Arquivo salvo: {nome_arquivo}")